In [1]:
!pip install scikit-image opencv-python tqdm


In [2]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.filters import threshold_otsu, threshold_sauvola
from skimage.morphology import remove_small_objects, remove_small_holes
from tqdm import tqdm


In [4]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [5]:
%%bash
kaggle datasets download nikhilroxtomar/brain-tumor-segmentation

Dataset URL: https://www.kaggle.com/datasets/nikhilroxtomar/brain-tumor-segmentation
License(s): unknown



100%|██████████| 312M/312M [00:00<00:00, 703MB/s]


In [6]:
def dice_score(pred, gt):
    pred = pred > 0
    gt = gt > 0
    intersection = np.logical_and(pred, gt).sum()
    return (2.0 * intersection) / (pred.sum() + gt.sum() + 1e-8)

def jaccard_score(pred, gt):
    pred = pred > 0
    gt = gt > 0
    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return intersection / (union + 1e-8)

In [12]:
dice_otsu = []
dice_sauvola = []
jac_otsu = []
jac_sauvola = []

# Unzip the dataset if it hasn't been already
!unzip -qo /content/brain-tumor-segmentation.zip -d /content/dataset

# --- Diagnostic step: List contents to verify path ---
print("--- Listing contents of /content/dataset ---")
!ls -R /content/dataset
print("------------------------------------------")
# --- End Diagnostic step ---

# Define the paths to the image and mask directories (these will be corrected after inspecting the directory structure)
IMAGE_DIR = '/content/dataset/images'
MASK_DIR = '/content/dataset/masks'

files = os.listdir(IMAGE_DIR)

for file in tqdm(files):

    img_path = os.path.join(IMAGE_DIR, file)
    mask_path = os.path.join(MASK_DIR, file)

    if not os.path.exists(mask_path):
        continue

    image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    # Ensure mask is binary
    _, gt_mask = cv2.threshold(gt_mask, 127, 255, cv2.THRESH_BINARY)

    # Smooth image
    image = cv2.GaussianBlur(image, (5,5), 0)

    # OTSU
    _, otsu_mask = cv2.threshold(
        image, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # SAUVOLA
    window_size = 25
    thresh_sauvola = threshold_sauvola(image, window_size=window_size)
    sauvola_mask = image > thresh_sauvola
    sauvola_mask = sauvola_mask.astype(np.uint8) * 255

    # Morphology cleanup
    kernel = np.ones((3,3), np.uint8)
    otsu_mask = cv2.morphologyEx(otsu_mask, cv2.MORPH_OPEN, kernel)
    otsu_mask = cv2.morphologyEx(otsu_mask, cv2.MORPH_CLOSE, kernel)

    sauvola_mask = cv2.morphologyEx(sauvola_mask, cv2.MORPH_OPEN, kernel)
    sauvola_mask = cv2.morphologyEx(sauvola_mask, cv2.MORPH_CLOSE, kernel)

    # Metrics
    dice_otsu.append(dice_score(otsu_mask, gt_mask))
    dice_sauvola.append(dice_score(sauvola_mask, gt_mask))

    jac_otsu.append(jaccard_score(otsu_mask, gt_mask))
    jac_sauvola.append(jaccard_score(sauvola_mask, gt_mask))

--- Listing contents of /content/dataset ---
/content/dataset:
images	masks

/content/dataset/images:
1000.png  1346.png  1691.png  2035.png	2380.png  2725.png  310.png  656.png
1001.png  1347.png  1692.png  2036.png	2381.png  2726.png  311.png  657.png
1002.png  1348.png  1693.png  2037.png	2382.png  2727.png  312.png  658.png
1003.png  1349.png  1694.png  2038.png	2383.png  2728.png  313.png  659.png
1004.png  134.png   1695.png  2039.png	2384.png  2729.png  314.png  65.png
1005.png  1350.png  1696.png  203.png	2385.png  272.png   315.png  660.png
1006.png  1351.png  1697.png  2040.png	2386.png  2730.png  316.png  661.png
1007.png  1352.png  1698.png  2041.png	2387.png  2731.png  317.png  662.png
1008.png  1353.png  1699.png  2042.png	2388.png  2732.png  318.png  663.png
1009.png  1354.png  169.png   2043.png	2389.png  2733.png  319.png  664.png
100.png   1355.png  16.png    2044.png	238.png   2734.png  31.png   665.png
1010.png  1356.png  1700.png  2045.png	2390.png  2735.png  320.p

100%|██████████| 3064/3064 [01:34<00:00, 32.54it/s]


In [13]:
print("\n===== FINAL RESULTS =====")
print("Average Dice (Otsu):", np.mean(dice_otsu))
print("Average Dice (Sauvola):", np.mean(dice_sauvola))
print("Average Jaccard (Otsu):", np.mean(jac_otsu))
print("Average Jaccard (Sauvola):", np.mean(jac_sauvola))


===== FINAL RESULTS =====
Average Dice (Otsu): 0.07051883259850633
Average Dice (Sauvola): 0.045429205371120614
Average Jaccard (Otsu): 0.03752119814811178
Average Jaccard (Sauvola): 0.023625176856761344
